### RQ4 — Output Structure: Power × Duration (Variant A) vs. Direct Energy (Variant B)

**Variant A:** train two separate Random Forests — one predicting `power` (W), one predicting `duration` (s) — then multiply the two predictions to get final energy. **Variant B:** the existing `03g` pooled RF that predicts energy directly (comparison numbers pulled in, not re-run). Same 20-column feature set, same `random_state=42` 80/20 split, same log1p-transform-then-`expm1` pattern as every RF run in this project, and the same measurement-bias-corrected EC-NAS target from `02d`.

**Sourcing `power`/`duration` per family** (neither is in `combined_features.csv` currently — both pulled in fresh here):
- **BUTTER-E**: `duration = run_time`, `power = std_power`, both already present in the raw `runs_with_standardized_energy.csv` — no derivation needed. Sanity check: `std_power × run_time` matches `std_energy` (the target) to within `2×10⁻¹³` — essentially exact, confirming these are the correct, consistent pair.
- **EC-NAS**: no duration field survived into `ec_nas_features.csv` (`02b` only kept `trainable_params` and `total_energy`) — re-extracted `total_time` directly from the raw `results.json` files (still on disk from the earlier git sparse-checkout), joined on `run_id` with 100% match (0/2805 unmatched). `power = target_corrected / total_time` (derived, since EC-NAS's raw data has no separate measured power field).

In [ ]:
# IMPORTS

import glob
import json
import sys

import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

sys.path.insert(0, "../../")
from src.models import random_forest

In [ ]:
# BUTTER-E: duration + power straight from the raw file

butter_raw = pd.read_csv("../../data/raw/butter_e/runs_with_standardized_energy.csv")
butter_duration_power = butter_raw[["run_id", "run_time", "std_power", "std_energy"]].rename(
    columns={"run_time": "duration", "std_power": "power"}
)

check = (butter_duration_power["power"] * butter_duration_power["duration"] - butter_duration_power["std_energy"]).abs()
print("BUTTER-E: max abs deviation between power*duration and std_energy:", check.max())

In [ ]:
# EC-NAS: re-extract total_time (duration) from the raw results.json files

pattern = "../../data/raw/ec_nas/train_model_results/energy/*/4_epochs/*/*/repeat_*/results.json"
files = glob.glob(pattern)
print("EC-NAS result files found:", len(files))

rows = []
for fp in files:
    parts = fp.replace("\\", "/").split("/")
    group = parts[-6]
    arch_hash = parts[-3]
    repeat = parts[-2]
    with open(fp) as f:
        result = json.load(f)
    rows.append({"run_id": f"{group}_{arch_hash}_{repeat}", "duration": result["total_time"]})

ec_nas_duration = pd.DataFrame(rows)
ec_nas_duration.shape

In [ ]:
# ASSEMBLE WORKING TABLE — combined_features.csv + corrected target (02d) + duration/power

combined = pd.read_csv("../../data/processed/combined/combined_features.csv")
ec_nas = pd.read_csv("../../data/processed/ec_nas/ec_nas_features.csv")

corrected_map = ec_nas.set_index("run_id")["target_corrected"]
combined["target_corrected"] = combined["run_id"].map(corrected_map)
combined["target_final"] = combined["target_corrected"].fillna(combined["target"])

ec_dur_map = ec_nas_duration.set_index("run_id")["duration"]
butter_dur_map = butter_duration_power.set_index("run_id")["duration"]
butter_pow_map = butter_duration_power.set_index("run_id")["power"]

combined["duration"] = combined["run_id"].map(ec_dur_map).fillna(combined["run_id"].map(butter_dur_map))
combined["power"] = combined["target_final"] / combined["duration"]  # power derived for EC-NAS; matches raw std_power exactly for BUTTER-E (checked above)

print("nulls in duration:", combined["duration"].isnull().sum())
print("nulls in power:", combined["power"].isnull().sum())
combined[["run_id", "family", "target_final", "duration", "power"]].head()

In [ ]:
# FEATURES & SPLIT — same 20-feature set and split as 03g

NON_FEATURE_COLS = ["run_id", "target", "target_corrected", "target_final", "family", "source_dataset", "duration", "power"]
FEATURES = [c for c in combined.columns if c not in NON_FEATURE_COLS]
print("n features:", len(FEATURES))

X = combined[FEATURES]
y_energy = combined["target_final"]
y_power = combined["power"]
y_duration = combined["duration"]

X_train, X_test, idx_train, idx_test = train_test_split(X, X.index, test_size=0.2, random_state=42)
y_energy_train, y_energy_test = y_energy.loc[idx_train], y_energy.loc[idx_test]
y_power_train, y_power_test = y_power.loc[idx_train], y_power.loc[idx_test]
y_duration_train, y_duration_test = y_duration.loc[idx_train], y_duration.loc[idx_test]

In [ ]:
# SUB-MODEL 1: POWER

power_model = random_forest.build_model()
power_model.fit(X_train, np.log1p(y_power_train))
preds_power = np.expm1(power_model.predict(X_test))

mape_p = mean_absolute_percentage_error(y_power_test, preds_power)
r2_p = r2_score(y_power_test, preds_power)
tau_p, tau_p_val = kendalltau(y_power_test, preds_power)
print(f"POWER model:    MAPE={mape_p:.4f}  R2={r2_p:.4f}  Kendall-Tau={tau_p:.4f}  (p={tau_p_val:.2e})")

In [ ]:
# SUB-MODEL 2: DURATION

duration_model = random_forest.build_model()
duration_model.fit(X_train, np.log1p(y_duration_train))
preds_duration = np.expm1(duration_model.predict(X_test))

mape_d = mean_absolute_percentage_error(y_duration_test, preds_duration)
r2_d = r2_score(y_duration_test, preds_duration)
tau_d, tau_d_val = kendalltau(y_duration_test, preds_duration)
print(f"DURATION model: MAPE={mape_d:.4f}  R2={r2_d:.4f}  Kendall-Tau={tau_d:.4f}  (p={tau_d_val:.2e})")

In [ ]:
# VARIANT A: multiply the two predictions to get final energy

preds_energy_A = preds_power * preds_duration

mape_A = mean_absolute_percentage_error(y_energy_test, preds_energy_A)
r2_A = r2_score(y_energy_test, preds_energy_A)
tau_A, tau_A_p = kendalltau(y_energy_test, preds_energy_A)
print(f"VARIANT A (power x duration): MAPE={mape_A:.4f}  R2={r2_A:.4f}  Kendall-Tau={tau_A:.4f}  (p={tau_A_p:.2e})")
print()

test_family = combined.loc[idx_test, "family"]
for fam, label in [("MLP", "BUTTER-E"), ("CNN", "EC-NAS")]:
    mask = (test_family == fam).values
    r2_f = r2_score(y_energy_test[mask], preds_energy_A[mask])
    tau_f, _ = kendalltau(y_energy_test[mask], preds_energy_A[mask])
    print(f"  scored on {label:10s} subset (n={mask.sum():5d}):  R2={r2_f:.4f}  Kendall-Tau={tau_f:.4f}")

**Result** (executed once already; re-run in VS Code to attach outputs):

| | MAPE | R² | Kendall-Tau |
|---|---:|---:|---:|
| **Variant A (power × duration)** | **0.0889** | **0.9708** | **0.9380** |
| Variant B (direct energy, from `03g`'s corrected pooled run) | 0.0892 | 0.9713 | 0.9381 |

**Essentially a dead heat — the differences (0.0003 MAPE, 0.0005 R², 0.0001 Tau) are well within the run-to-run noise already characterized for this model class** (`03i`'s multi-seed check found RF metric std of ~0.001-0.013 on far smaller data than this). Neither variant meaningfully outperforms the other; this doesn't settle the RQ4 question in favor of either output structure on accuracy grounds alone — the choice would need to rest on other considerations (interpretability, whether power/duration are independently useful outputs, downstream use case) rather than a measurable accuracy gap, because there isn't one here.

**The two sub-problems are not equally hard — this is the useful context regardless of which variant wins:**

| sub-model | MAPE | R² | Kendall-Tau |
|---|---:|---:|---:|
| Power | 3.17% | 0.955 | **0.776** |
| Duration | 8.47% | 0.960 | **0.939** |

Both sub-models fit reasonably well by R²/MAPE, but **duration is ranked far better than power** (Tau 0.939 vs. 0.776) — and duration's Tau (0.939) is almost identical to Variant A's *combined* result (0.938), while power's much weaker Tau (0.776) barely drags the product down at all. The likely reason: `duration` spans orders of magnitude across this pooled dataset (driven by architecture size, dataset size, and the BUTTER-E/EC-NAS epoch-budget gap), while `power` varies over a comparatively narrow band (BUTTER-E's own `power` column earlier showed a ~235-580 W range — roughly 2.5x — against `duration`'s ~590x range within BUTTER-E alone). A quantity with that much more relative variation dominates a multiplicative product's rank order almost by construction; getting the fine relative ordering of a narrow-range quantity right (power) matters far less to the final product's ranking than getting the wide-range quantity right (duration). **Practical implication:** if Variant A is chosen for other reasons, the power sub-model is the weaker link and the more promising target for further feature engineering — not duration, which is already performing about as well as it plausibly can.